Rag pipeline

In [ ]:
import os
from langchain_community.document_loaders import PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [6]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents) #Stores multiple pdf into one
            print(f"Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("textfile")

Found 1 PDF files to process

Processing: main.pdf
Loaded 17 pages

Total documents loaded: 17


In [11]:
#Chunking
def spli_documents(documents , chunk_sizes = 1000 , chunk_overlaps = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_sizes ,
        chunk_overlap = chunk_overlaps,
        separators=["\n\n", "\n", " ", ""]
    )
    split_doc = text_splitter.split_documents(documents)
    print(f"The document length : {len(documents)} splitted into : {len(split_doc)} chunks")


In [ ]:
chunks = spli_documents(all_pdf_documents)


The document length : 17 splitted into : 41 chunks


Embedding and Vectordb


In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from sklearn.metrics.pairwise import cosine_similarity
from typing import List , Dict  , Any , Tuple

In [18]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): #Open source model
        
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"{self.model_name} is loading")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded sucessffully")
        except Exception as e:
            print("Error loading model")
            raise
    
    def generate_embedding(self , texts : List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not Loaded")
        
        print("Letsssssss goooo")
        embeddings = self.model.encode(texts , show_progress_bar=True)
        print("Embedding Completed")
        return embeddings

In [ ]:
embedding = EmbeddingManager() #Intialize the embedding manager
embedding

all-MiniLM-L6-v2 is loading
Model Loaded sucessffully


VectorDB


In [ ]:
class VectorStore:
    def __init__(self , collection_name : str = "pdf_documents" , persist_directory : str = "vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory , exist_ok=True) #Initializes the directory
            self.client = chromadb.PersistentClient(path = self.persist_directory)  #initializes the chromadb

            self.collection = self.client.get_or_create_collection(
                name  = self.collection_name, #Cha bhaney tehi db use garcha chaina bhaney new create garcha
            )

            print(f"The Database initialized with {self.collection_name}")
        except Exception as e: 
            print("Error initliazing the Database")
            raise
    
    def add_documents(self , document : List[Any] , embeddings : np.ndarray):
        if len(document) != len(embeddings):
            raise ValueError("Number of the document is not equal to the Embbedings")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(document, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(document)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

